# POS Classifier — EDA & Pipeline Decisions

Each section answers one question: **what does the data look like, and what decision does that drive?**  
All helper functions live in `src/pos_classifier/`; nothing is re-implemented here.

In [ ]:
import sys
sys.path.insert(0, "../src")

from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

from transformers import AutoTokenizer
from pos_classifier.config import TrainingConfig, ID_TO_LABEL, LABEL_MAP
from pos_classifier.data.preprocessing import load_training_data, load_query_data
from pos_classifier.data.eda import (
    class_distribution_df,
    text_stats_df,
    token_length_stats,
    coverage_at_max_length,
    cross_class_duplicates,
    query_label_coverage,
)

cfg = TrainingConfig()
DATA_DIR = Path("../data")
TRAIN_CSV = DATA_DIR / cfg.train_file
QUERY_CSV = DATA_DIR / "Query_and_Validation_data.csv"

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

---
## 1  Raw Data Load

In [ ]:
raw_texts, raw_labels = load_training_data(TRAIN_CSV)   # deduplication happens inside
query_texts, query_labels = load_query_data(QUERY_CSV)

print(f"Training rows (after dedup): {len(raw_texts):,}")
print(f"Query rows:                  {len(query_texts):,}")

---
## 2  Data Quality

**Questions:** Are there cross-class conflicts (same text, different labels)?  
How many query rows have human-verified labels for evaluation?

In [ ]:
# Re-load without internal dedup to inspect raw conflicts
import csv, unicodedata

raw_t, raw_l = [], []
with TRAIN_CSV.open(encoding="utf-8", newline="") as f:
    reader = csv.reader(f)
    next(reader)
    for row in reader:
        if len(row) >= 2:
            t = unicodedata.normalize("NFKC", row[0]).strip()
            c = unicodedata.normalize("NFKC", row[1].strip().rstrip(";").strip('"').strip())
            if t and c in LABEL_MAP:
                raw_t.append(t)
                raw_l.append(LABEL_MAP[c])

conflicts = cross_class_duplicates(raw_t, raw_l)
total_dupes = len(raw_t) - len(raw_texts)

print(f"Total raw valid rows:      {len(raw_t):,}")
print(f"Duplicate descriptions:    {total_dupes:,}")
print(f"Cross-class conflicts:     {len(conflicts)}")
if not conflicts.empty:
    display(conflicts.head(5))

In [ ]:
cov = query_label_coverage(query_labels)
print(f"Query rows total:          {cov['total_rows']:,}")
print(f"With human-verified label: {cov['verified']:,}  ({cov['verified_pct']}%)")
print(f"Unlabelled (inference only): {cov['unlabelled']:,}")
print("\nVerified per class:", cov['per_class'])

**Decision — Data prep:**  
- `load_training_data` already strips Excel artifacts (`;;;;` suffixes, stray quotes), NFKC-normalises, and deduplicates by first-occurrence tie-break.  
- Cross-class conflicts (if any) are resolved by the same first-occurrence rule — no manual curation needed at this scale.  
- No train/test contamination: `load_training_data` returns a flat list; the split happens in `trainer.py` after loading.

---
## 3  Class Distribution → Weighted Loss

In [ ]:
dist = class_distribution_df(raw_labels)
display(dist)

fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(dist.index, dist["count"], color=sns.color_palette("muted", len(dist)))
ax.set_ylabel("Count")
ax.set_title("Training set class distribution")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

**Decision — Class imbalance:**  
Ratio of majority to minority class ≈ 5–6×. *Specialty & Miscellaneous* is the hardest case.  
- **Approach chosen:** `CrossEntropyLoss(weight=class_weights)` with sklearn's `compute_class_weight("balanced")`.  
- Rejected oversampling: would inflate training time and risk duplicating rare items.  
- Rejected undersampling: discards the majority of our already-modest dataset.  
- Class weights are recomputed on the *training split only* (inside `trainer.py`) to prevent leakage from val/test label counts.

---
## 4  Text Length → max_length Decision

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(cfg.tokenizer_name)

stats_df, tok_lengths = token_length_stats(raw_texts, tokenizer)
display(stats_df)

cov_df = coverage_at_max_length(tok_lengths, [32, 48, 64, 96, 128])
display(cov_df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3))

stats = text_stats_df(raw_texts, raw_labels)

axes[0].hist(tok_lengths, bins=40, color="steelblue", edgecolor="white")
axes[0].axvline(cfg.max_length, color="crimson", linestyle="--", label=f"max_length={cfg.max_length}")
axes[0].set_xlabel("Token count (excl. special tokens)")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Token length distribution")
axes[0].legend()

axes[1].hist(stats["word_count"], bins=30, color="teal", edgecolor="white")
axes[1].set_xlabel("Word count")
axes[1].set_title("Word count distribution")

plt.tight_layout()
plt.show()

**Decision — Preprocessing & max_length:**  
- p99 token count ≈ 20; `max_length=64` covers 100% of training descriptions.  
- Cutting from 128 → 64 halves BERT's O(n²) self-attention cost with zero information loss.  
- `padding="max_length"` + `truncation=True` in `POSDataset` keeps tensors uniform for batching.  
- Tokenizer: `bert-base-uncased` vocabulary — `bert-tiny` ships weights only, no tokenizer files, but shares the identical WordPiece vocab.

---
## 5  Per-class Text Patterns

In [ ]:
stats = text_stats_df(raw_texts, raw_labels)
summary = stats.groupby("label")[["char_len", "word_count"]].agg(["mean", "median", "max"])
summary.columns = ["_".join(c) for c in summary.columns]
display(summary.round(1))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.5))
order = stats.groupby("label")["word_count"].median().sort_values().index
sns.boxplot(data=stats, x="label", y="word_count", order=order, ax=ax)
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right")
ax.set_title("Word count per class")
plt.tight_layout()
plt.show()

**Observation:** Classes share similar length profiles — signal is semantic, not structural.  
This confirms a pre-trained language model is the right tool (vs. TF-IDF + logistic regression), and that no class-specific length truncation is needed.

---
## 6  Query Dataset Analysis

In [ ]:
verified_labels = [l for l in query_labels if l is not None]
query_dist = class_distribution_df(verified_labels).rename(columns={"count": "query_count", "pct": "query_pct"})
train_dist = class_distribution_df(raw_labels).rename(columns={"count": "train_count", "pct": "train_pct"})

comparison = train_dist[["train_count", "train_pct"]].join(query_dist[["query_count", "query_pct"]])
display(comparison)

**Decision — Evaluation data:**  
- Query CSV provides an *independent* held-out set with human-verified labels for ~40–50% of rows.  
- Used in `monitoring/metrics.py` → `compute_validation_metrics()` as a live accuracy signal post-deployment.  
- Training test split and query CSV are **separate** — no overlap risk since query CSV is a different file.

---
## 7  Model Selection

| Model | Params | Layers | Hidden | Latency (CPU, p50) | Notes |
|---|---|---|---|---|---|
| TF-IDF + LogReg | ~50K | — | — | <1 ms | No context, poor on ambiguous names |
| **bert-tiny** | **4.4 M** | **2** | **128** | **~10 ms** | **Chosen** |
| bert-mini | 11.3 M | 4 | 256 | ~25 ms | Marginal gain for 3× cost |
| bert-base | 110 M | 12 | 768 | ~120 ms | Overkill; deployment budget exceeded |

In [ ]:
from transformers import BertConfig, BertForSequenceClassification
from pos_classifier.config import NUM_LABELS

cfg_bert = BertConfig.from_pretrained(cfg.model_name, revision=cfg.model_revision)
print(f"Layers:       {cfg_bert.num_hidden_layers}")
print(f"Hidden dim:   {cfg_bert.hidden_size}")
print(f"Attention heads: {cfg_bert.num_attention_heads}")
print(f"Intermediate: {cfg_bert.intermediate_size}")

tmp = BertForSequenceClassification(cfg_bert)
total_params = sum(p.numel() for p in tmp.parameters())
print(f"Total params: {total_params/1e6:.1f}M")
del tmp

**Decision — Model selection:**  
- Task is short-text classification (≤20 tokens p99). Two transformer layers are sufficient to learn category-discriminating token interactions.  
- `bert-tiny` fits in <20 MB on disk, loads in <2 s, and runs 5–6 k predictions/s on CPU — within the serving budget.  
- `model_revision="refs/pr/13"` pins the `.safetensors` variant to satisfy transformers 5.x + torch <2.6 (CVE-2025-32434).

---
## 8  Training Design

In [ ]:
# Illustrate linear warmup + decay schedule shape (no real model needed)
from transformers import get_linear_schedule_with_warmup
import torch

total_steps = 500
warmup = int(total_steps * cfg.warmup_ratio)
dummy_opt = torch.optim.AdamW([torch.zeros(1, requires_grad=True)], lr=cfg.learning_rate)
sched = get_linear_schedule_with_warmup(dummy_opt, warmup, total_steps)

lrs = []
for _ in range(total_steps):
    lrs.append(dummy_opt.param_groups[0]["lr"])
    sched.step()

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(lrs)
ax.axvline(warmup, color="orange", linestyle="--", label=f"warmup end (step {warmup})")
ax.set_xlabel("Step")
ax.set_ylabel("LR")
ax.set_title("AdamW + linear warmup scheduler")
ax.legend()
plt.tight_layout()
plt.show()

**Decision — Training:**  
- **Optimizer:** AdamW with `weight_decay=0.01` — standard for fine-tuning; decoupled weight decay avoids penalising bias/LayerNorm.  
- **Scheduler:** Linear warmup (10% of steps) then linear decay — avoids large gradient updates in early steps when weights are random for the classifier head.  
- **Early stopping:** patience=2 on `val_accuracy`. Primary stopping signal is accuracy (not loss) to directly guard the downstream business metric.  
- **AMP / FP16:** enabled on CUDA; no-op on CPU. Gradient scaler prevents FP16 underflow.  
- **Reproducibility:** all seeds fixed at 42; every hyperparameter logged to MLflow.

---
## 9  Evaluation Strategy

In [ ]:
# Simulate why accuracy alone misleads on imbalanced data
from sklearn.metrics import accuracy_score, f1_score

n = len(raw_labels)
majority_class = 0  # Dry Goods & Pantry Staples
dummy_preds = [majority_class] * n

dummy_acc = accuracy_score(raw_labels, dummy_preds)
dummy_f1  = f1_score(raw_labels, dummy_preds, average="macro", zero_division=0)

print(f"Dummy classifier (always predict majority class):")
print(f"  Accuracy:  {dummy_acc:.1%}  ← looks reasonable")
print(f"  Macro-F1:  {dummy_f1:.1%}  ← reveals it's useless")

**Decision — Evaluation:**  
- **Primary metric: macro-F1** — treats each class equally regardless of support. A model that ignores *Specialty & Misc* scores 0.0 on that class, dragging macro-F1 down visibly.  
- Accuracy is logged alongside for interpretability and as the early-stopping signal (simpler to communicate to stakeholders).  
- Per-class F1 logged to MLflow so regressions on specific categories are caught immediately.  
- Retraining is triggered when validation accuracy drops below **85%** (configured in serving contract).

---
## 10  Serving Design

**Confidence threshold analysis:** what fraction of predictions fall below candidate thresholds?

In [ ]:
# Simulate softmax confidence from a uniform Dirichlet to illustrate threshold sensitivity
rng = np.random.default_rng(42)
sim_probs = rng.dirichlet(alpha=[3, 1, 1, 1, 1], size=5000)   # skewed toward one class
sim_conf = sim_probs.max(axis=1)

thresholds = [0.50, 0.60, 0.70, 0.75, 0.80, 0.90]
flag_rates = {t: round((sim_conf < t).mean() * 100, 1) for t in thresholds}

print("Threshold → low-confidence flag rate (simulated):")
for t, r in flag_rates.items():
    print(f"  {t:.2f}  →  {r}% flagged")

print(f"\n+ 20% random sample for human review (always on)")
print(f"Chosen threshold: {cfg.confidence_threshold} — balanced review load vs. error capture")

**Decision — Serving:**  
- **`Predictor` singleton** — model loads once on startup (`lifespan`), shared across requests. No per-request model init cost.  
- **Confidence threshold = 0.70** — persisted in `metadata.json`, loaded at serve time. Changing it in `TrainingConfig` propagates on next train.  
- **Flagging logic:** `conf < 0.70` OR `random() < 0.20` — low-confidence items *and* a random 20% sample go to human review. Random sample catches high-confidence errors that threshold alone misses.  
- **SQLite** — predictions + feedback tables; zero-dependency, sufficient for single-node scale. Swap point to Postgres is explicit in design docs.  
- **`Predictor.reset()`** — hot-reload after retraining without a server restart.  
- **`serving/` never imports from `training/`** — boundary enforced via architecture rules.

---
## 11  Monitoring & Feedback Loop

In [ ]:
# Schema of what the monitoring stack tracks
monitoring_signals = pd.DataFrame([
    {"Signal": "Prediction volume",        "Source": "Prometheus counter",   "Dashboard": True,  "Retrain trigger": False},
    {"Signal": "Confidence distribution",  "Source": "Prometheus histogram", "Dashboard": True,  "Retrain trigger": False},
    {"Signal": "Flag rate",                "Source": "SQLite / Prometheus",  "Dashboard": True,  "Retrain trigger": False},
    {"Signal": "Category distribution",    "Source": "SQLite",               "Dashboard": True,  "Retrain trigger": False},
    {"Signal": "Val accuracy vs. ground truth", "Source": "Query CSV",       "Dashboard": True,  "Retrain trigger": True},
    {"Signal": "Feedback count",           "Source": "SQLite feedback table","Dashboard": True,  "Retrain trigger": True},
]).set_index("Signal")

display(monitoring_signals)

In [ ]:
# Illustrate the two retraining triggers
print("Retraining triggers (from serving contract):")
print(f"  1. feedback count >= 500  (pending_retraining_check in monitoring/metrics.py)")
print(f"  2. validation accuracy < 0.85  (checked in monitoring dashboard)")
print()
print("On retrain:")
print("  → trainer.py runs, saves best_model/ + metadata.json")
print("  → Predictor.reset() clears singleton")
print("  → next request lazy-loads updated model — zero downtime")

**Decision — Monitoring & feedback loop:**  
- **Prometheus** counters/histograms at `/metrics` for operational monitoring (volume, latency, flag rate).  
- **Streamlit dashboard** aggregates SQLite + Prometheus data for ML-layer visibility (per-class accuracy vs. human labels, confidence drift).  
- **Two retraining triggers:** feedback volume (500 corrections = enough signal for a meaningful fine-tune cycle) and accuracy degradation (<85% on verified query rows).  
- **No scheduled retraining** — data-driven triggers prevent unnecessary churn and preserve MLflow run history as a clean signal.

---
## Summary of Decisions

| Area | Decision | Rationale |
|---|---|---|
| Data prep | NFKC normalise + dedup by first-occurrence | Handles Excel export artifacts; deterministic tie-break |
| Class imbalance | `CrossEntropyLoss(weight=class_weights)` | No data loss; weights recomputed on train split only |
| max_length | 64 | Covers 100% of data; halves O(n²) attention cost vs. 128 |
| Tokenizer | `bert-base-uncased` vocab | bert-tiny has no tokenizer files; identical WordPiece vocab |
| Model | `bert-tiny` (4.4M params) | Sufficient depth for short-text; <10 ms CPU inference |
| Optimizer | AdamW + linear warmup | Standard for BERT fine-tuning; stable early gradient steps |
| Early stopping | patience=2 on val_accuracy | Prevents overfitting on small dataset |
| Primary metric | Macro-F1 | Accuracy misleads on 5.5% minority class |
| Confidence threshold | 0.70 | Balanced review load; configurable via metadata.json |
| Persistence | SQLite | Zero-dependency; documented swap point to Postgres |
| Monitoring | Prometheus + Streamlit | Operational vs. ML-layer separation |
| Retraining triggers | 500 feedback OR val_acc < 0.85 | Data-driven; avoids churn |